# Sentiment Analysis on IMDB: LSTM (From Scratch) vs. Pretrained BERT

This notebook runs the end-to-end training, evaluation, and comparison using the modular package in `src/sentiment_lstm_vs_bert/`:
1. **DataLoaders:** Load prepared splits (`data/imdb_prepared`) for both LSTM and BERT.
2. **LSTM from Scratch:** Train `SentimentLSTM` and evaluate on `test`.
3. **Pretrained BERT:** Fine-tune `bert-base-uncased` and evaluate on `test`.
4. **Comparison:** Compare metrics, training/inference speed, confusion matrices, and ROC curves.
5. **Error Analysis:** Inspect reviews where BERT and LSTM disagree.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import ConfusionMatrixDisplay, roc_curve
from transformers import AutoModelForSequenceClassification

from sentiment_lstm_vs_bert import (
    LSTMConfig,
    SentimentLSTM,
    TrainConfig,
    build_bert_dataloaders,
    build_lstm_dataloaders,
    evaluate,
    load_prepared_dataset,
    train_model,
)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR = Path("data/imdb_prepared")
CKPT_DIR = Path("artifacts/checkpoints")
RESULTS_DIR = Path("artifacts/results")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
print(f"Using device: {DEVICE}")

## 1. Build DataLoaders (LSTM & BERT)


In [ ]:
MAX_LEN = 256

# LSTM DataLoaders
lstm_train_dl, lstm_val_dl, lstm_test_dl, vocab_payload = build_lstm_dataloaders(
    data_dir=DATA_DIR,
    max_len=MAX_LEN,
    batch_size=64,
)

# BERT DataLoaders (use 'distilbert-base-uncased' if you want faster CPU runs)
BERT_MODEL_NAME = "bert-base-uncased"
bert_train_dl, bert_val_dl, bert_test_dl, bert_tokenizer = build_bert_dataloaders(
    data_dir=DATA_DIR,
    model_name=BERT_MODEL_NAME,
    max_len=MAX_LEN,
    batch_size=16,
)

print(f"LSTM vocab size: {len(vocab_payload['token_to_idx']):,}")
print(f"LSTM batches -> train: {len(lstm_train_dl)}, val: {len(lstm_val_dl)}, test: {len(lstm_test_dl)}")
print(f"BERT batches -> train: {len(bert_train_dl)}, val: {len(bert_val_dl)}, test: {len(bert_test_dl)}")

## 2. Train & Evaluate LSTM (From Scratch)


In [ ]:
lstm_config = LSTMConfig(
    vocab_size=len(vocab_payload["token_to_idx"]),
    embed_dim=128,
    hidden_dim=128,
    num_layers=2,
    bidirectional=True,
    dropout=0.3,
    num_classes=2,
    pad_idx=vocab_payload["pad_id"],
    pooling="last",
)
lstm_model = SentimentLSTM(lstm_config)
lstm_params = lstm_model.count_parameters()
print(f"SentimentLSTM trainable parameters: {lstm_params:,}")

lstm_train_out = train_model(
    lstm_model,
    lstm_train_dl,
    lstm_val_dl,
    TrainConfig(
        epochs=5,
        lr=1e-3,
        weight_decay=1e-4,
        max_grad_norm=1.0,
        save_path=CKPT_DIR / "lstm_best.pt",
        device=DEVICE,
    ),
)

lstm_test = evaluate(lstm_model, lstm_test_dl, device=DEVICE, return_predictions=True)
print(
    f"\nLSTM Test -> Acc: {lstm_test['accuracy']:.4f} | "
    f"F1: {lstm_test['f1']:.4f} | ROC-AUC: {lstm_test['roc_auc']:.4f} | "
    f"Latency: {lstm_test['latency_ms_per_sample']:.3f} ms/sample"
)

In [ ]:
def plot_history(history: list[dict], title: str) -> None:
    df = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    axes[0].plot(df["epoch"], df["train_loss"], marker="o", label="Train Loss")
    axes[0].plot(df["epoch"], df["val_loss"], marker="o", label="Val Loss")
    axes[0].set_title(f"{title} — Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(df["epoch"], df["val_acc"], marker="o", label="Val Accuracy")
    axes[1].plot(df["epoch"], df["val_f1"], marker="o", label="Val F1")
    axes[1].set_title(f"{title} — Validation Metrics")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

plot_history(lstm_train_out["history"], "LSTM (From Scratch)")

## 3. Fine-Tune & Evaluate Pretrained BERT


In [ ]:
bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME,
    num_labels=2,
)
bert_params = sum(p.numel() for p in bert_model.parameters() if p.requires_grad)
print(f"{BERT_MODEL_NAME} trainable parameters: {bert_params:,}")

bert_train_out = train_model(
    bert_model,
    bert_train_dl,
    bert_val_dl,
    TrainConfig(
        epochs=3,
        lr=2e-5,
        weight_decay=0.01,
        max_grad_norm=1.0,
        save_path=CKPT_DIR / "bert_best.pt",
        device=DEVICE,
    ),
)

bert_test = evaluate(bert_model, bert_test_dl, device=DEVICE, return_predictions=True)
print(
    f"\nBERT Test -> Acc: {bert_test['accuracy']:.4f} | "
    f"F1: {bert_test['f1']:.4f} | ROC-AUC: {bert_test['roc_auc']:.4f} | "
    f"Latency: {bert_test['latency_ms_per_sample']:.3f} ms/sample"
)

In [ ]:
plot_history(bert_train_out["history"], f"BERT ({BERT_MODEL_NAME})")

## 4. Side-by-Side Model Comparison


In [ ]:
comparison_df = pd.DataFrame([
    {
        "Model": "LSTM (From Scratch)",
        "Parameters": lstm_params,
        "Train Time (s)": lstm_train_out["total_train_time_s"],
        "Test Loss": round(lstm_test["loss"], 4),
        "Accuracy": round(lstm_test["accuracy"], 4),
        "Precision": round(lstm_test["precision"], 4),
        "Recall": round(lstm_test["recall"], 4),
        "F1": round(lstm_test["f1"], 4),
        "ROC-AUC": round(lstm_test["roc_auc"], 4),
        "Latency (ms/sample)": lstm_test["latency_ms_per_sample"],
    },
    {
        "Model": f"BERT ({BERT_MODEL_NAME})",
        "Parameters": bert_params,
        "Train Time (s)": bert_train_out["total_train_time_s"],
        "Test Loss": round(bert_test["loss"], 4),
        "Accuracy": round(bert_test["accuracy"], 4),
        "Precision": round(bert_test["precision"], 4),
        "Recall": round(bert_test["recall"], 4),
        "F1": round(bert_test["f1"], 4),
        "ROC-AUC": round(bert_test["roc_auc"], 4),
        "Latency (ms/sample)": bert_test["latency_ms_per_sample"],
    },
])
comparison_df.to_csv(RESULTS_DIR / "comparison_metrics.csv", index=False)
comparison_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, title, res in zip(
    axes[:2],
    ["LSTM Confusion Matrix", "BERT Confusion Matrix"],
    [lstm_test, bert_test],
):
    cm = np.array(res["confusion_matrix"])
    disp = ConfusionMatrixDisplay(cm, display_labels=["neg", "pos"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues", values_format="d")
    ax.set_title(title)

# ROC Curves
for name, res in [("LSTM", lstm_test), ("BERT", bert_test)]:
    fpr, tpr, _ = roc_curve(res["y_true"], res["y_prob"])
    axes[2].plot(fpr, tpr, label=f"{name} (AUC = {res['roc_auc']:.4f})")

axes[2].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[2].set_title("ROC Curve Comparison")
axes[2].set_xlabel("False Positive Rate")
axes[2].set_ylabel("True Positive Rate")
axes[2].legend()

plt.tight_layout()
plt.show()

## 5. Qualitative Error Analysis


In [ ]:
test_texts = load_prepared_dataset(DATA_DIR)["test"]["text"]
y_true = lstm_test["y_true"]
lstm_ok = lstm_test["y_pred"] == y_true
bert_ok = bert_test["y_pred"] == y_true

print(f"Both correct:          {(lstm_ok & bert_ok).sum():,}")
print(f"Both wrong:            {(~lstm_ok & ~bert_ok).sum():,}")
print(f"BERT right, LSTM wrong:{(~lstm_ok & bert_ok).sum():,}")
print(f"LSTM right, BERT wrong:{(lstm_ok & ~bert_ok).sum():,}")

bert_wins_idx = np.where((~lstm_ok) & bert_ok)[0][:3]
print("\n--- Examples where BERT succeeded and LSTM failed ---")
for idx in bert_wins_idx:
    label_str = "pos" if y_true[idx] == 1 else "neg"
    print(f"[Idx {idx} | True: {label_str} | LSTM prob={lstm_test['y_prob'][idx]:.2f} | BERT prob={bert_test['y_prob'][idx]:.2f}]")
    print(test_texts[int(idx)][:350], "...\n")